In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("IMDB Dataset.csv")

In [ ]:
df.shape

In [ ]:
df.head(5)

In [ ]:
df.isnull().sum()

In [ ]:
df.drop_duplicates(inplace = True)

In [ ]:
df.shape

# Pre-Processing

### 1. Converting to lowercase 

In [ ]:
df["review"] = df["review"].str.lower()

In [ ]:
df["review"]

### Removing the URLs

In [ ]:
import re

In [ ]:
def remove_urls(text):
    text = re.sub(r"http\S+" ,"", text)
    return text

df["review"] = df["review"].apply(remove_urls)

## Removing punctuations

In [ ]:
def remove_puncs(text):
    text = re.sub(r"[^A-Za-z0-9\s]", "", text)
    return text
    
df["review"] = df["review"].apply(remove_puncs)

In [ ]:
df.head(5)

### Removing HTML

In [ ]:
def remove_html(text):
    text = re.sub(r"<.*?>", "", text)
    return text

df["review"] = df["review"].apply(remove_html)

In [ ]:
df.head(5)

### Removing stopwords

In [ ]:
import nltk

# nltk.download("punkt")
# nltk.download("punkt_tab")
# nltk.download("stopwords")

In [ ]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [ ]:
# sample_text = "I like coding in Python!"
# tokens = word_tokenize(sample_text)
# tokens

In [ ]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text = text.replace(word, "")
    return text

df["review"] = df["review"].apply(remove_stopwords)

In [ ]:
df.head(5)

### Stemming

In [ ]:
from nltk.stem import PorterStemmer 

In [ ]:
def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)
    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)

In [ ]:
df.head(5)

### Encoding

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["sentiment"] = le.fit_transform(df["sentiment"])

y = df["sentiment"]


In [ ]:
y

### Vectorization

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)

X = tf.fit_transform(df["review"])

### Dataset & DataLoaders

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state = 42
)

In [ ]:
X_train.shape

In [ ]:
X_test.shape

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
# Converting sprase matrix to numpy array
X_train = X_train.toarray()
X_test = X_test.toarray()

In [ ]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [ ]:
train_loader = DataLoader(train_set, shuffle = True, batch_size = 64)
test_loader = DataLoader(test_set, shuffle = True, batch_size = 64)

# Build Our RNN

In [ ]:
class RNN(nn.Module):
  def __init__(self, input_size, hidden_size = 128, num_layers = 1 ):
    super().__init__()

    self.hidden_size = hidden_size
    self.num_layers = num_layers

    # RNN Layer
    self.rnn = nn.RNN(
        input_size, hidden_size, num_layers, batch_first = True
    )

    # Fully connected layer
    self.fc = nn.Linear(hidden_size, 1)


  def forward(self, x):

    # optional => shape (num of layers, batch_size, hidden size)
    h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

    out, _ = self.rnn(x, h0)
    
    # 1st value = hidden state of all the timesteps => (batch, seq_len, hidden_size)
    # 2nd value = final hidden stae of last timestep

    out = self.fc(out[:, -1, :])
    return out


In [ ]:
input_size = X_train.shape[1]
model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

# Training the RNN

In [ ]:
epochs = 100
for epoch in range(epochs):
  total_loss = 0
  model.train()

  for Xb, yb in train_loader:
    optimizer.zero_grad()

    Xb = Xb.unsqueeze(1)   # add singleton direction
    outputs = model(Xb)     # (batch_size, 1)
    
    outputs = torch.sigmoid(outputs.squeeze())   #batch_size => probability
    
    loss = criterion(outputs, yb)   # compute loss
    loss.backward()  # backprop
    optimizer.step() # update weights
    
    total_loss += loss.item()
  
  avg_loss = total_loss / len(train_loader)
  print(f"epoch = {epoch+1}/{epochs} and loss = {avg_loss:.4f}")

# Evaluate

In [ ]:
# Evaluate

model.eval()
with torch.no_grad():
  correct_vals = 0
  tot_vals = 0

  for Xb, yb in test_loader:
    Xb = Xb.unsqueeze(1)

    outputs = model(Xb)
    predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

    tot_vals +=  yb.size(0)
    correct_vals += (predicted == yb).sum().item()
  accuracy = correct_vals / tot_vals

  print(f"Accuracy = {accuracy*100:.2f}%")
